# NB08 — Build03-B Q1–Q10 Full Benchmark & Build04 Gap Analysis

**Scientific AI Builder · Regulatory Investigation Agent · Frozen Benchmark v1**

**Purpose:** Evaluate the *existing* Build03-B (local `search_regulation()` + official Cellar `check_current_law()`) across all frozen Q1–Q10 questions; reuse NB04b frozen Build02 RAG answers as baseline; inspect failures to prioritize Build04 functionality. **Not** a new 3-arm ablation; NB07 already tested Q8 A/B.

**Controls**
- Same five-law frozen corpus (1,383 chunks), exact NB03 dense/BM25/RRF retriever and 50/50 ordered retrieval parity.
- Same Q1–Q10 text, shared scenario, `gpt-6-luna`, `medium`, and three-heading answer format.
- Build02 baseline **read-only**; Build03-B performs fresh individual runs. Original NB07 Q8 and NB05 Q7 are *historical pilots*, not silently substituted into a full-benchmark result.
- **Important limitation:** NB07 A/B tool descriptions and `MATCHED_RULES` are reused, but Build02 prompt/construction is not identical to Agent instructions; compare as staged-system evaluation rather than causal proof.
- No paid run by default; controlled run flags must be enabled explicitly tomorrow.

## 0. Setup
Run from the `Regulatory_agent` repository root, containing `data/`. Dependencies are the same as NB07. **No OpenAI API key is needed for setup or parity.**

In [ ]:
# Only when packages are missing:
# %pip install openai-agents sentence-transformers rank-bm25 numpy pandas requests beautifulsoup4

In [1]:
import os, re, json, hashlib, platform
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from agents import Agent, Runner, ModelSettings, function_tool

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
print("Python:", platform.python_version(), "| Model:", MODEL_NAME)
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))
print("API key is not required until the Q8 official run.")

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | Model: gpt-6-luna
API key available: False
API key is not required until the Q8 official run.


## 1. Load frozen assets and reference outputs
All references are loaded from existing files. Original NB04b answer CSV is never regenerated or overwritten.

In [2]:
FILES = {
    "chunks": __import__("pathlib").Path("data/processed/chunks_legal_aware.json"),
    "benchmark": __import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json"),
    "retrieval_config": __import__("pathlib").Path("data/processed/retrieval_config_nb03_final_hybrid_static_rag.json"),
    "nb03_results": __import__("pathlib").Path("data/processed/retrieval_results_nb03_final_hybrid_static_rag.csv"),
    "nb04b_config": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_experiment_config.json"),
    "nb04b_answers": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_answers_comparison.csv"),
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
Q8 = questions["Q8"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

FROZEN_Q8_ANSWER = comparison_df.loc[comparison_df.qid=="Q8", "rag_answer"].iloc[0]
print("Benchmark:", benchmark["benchmark_id"])
print("Chunks:", len(legal_chunks), "| Q8:", Q8)
print("Frozen NB04b Build02 RAG answers available:", comparison_df.qid.nunique())

Benchmark: regulatory_agent_v1_notion_2026-10-08
Chunks: 1383 | Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?
Frozen NB04b Build02 RAG answers available: 10


## 2. Exact frozen NB03 hybrid retriever
Keep searchable text, tokenizer, MiniLM embeddings, BM25, RRF=60, candidate depth=50, max-2 per source-group and top-5 unchanged.

In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

## 3. STOP-GATE: ordered NB03 retrieval parity 50/50
Stop immediately if chunk IDs or rank order differ. This proves baseline retrieval reproducibility for the *benchmark question queries*, not necessarily every dynamic agent query.

In [5]:
PARITY_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb08_retrieval_parity_audit.csv")
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb06_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official Q8 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build03_nb08_retrieval_parity_audit.csv


## 4. Frozen law registry + Cellar SPARQL client
Only allowed five law IDs; official metadata lookup cannot prove article-level applicability. Includes the `FILTER(CONTAINS(...))` correction from NB06 v3.

In [6]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (__import__("pathlib").Path("data/legal_sources") / item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


In [7]:
# Official Cellar SPARQL: CELEX identifiers are stored under
# cdm:resource_legal_id_celex (NOT cdm:work_id_document).
# No EUR-Lex HTML scraping, no user-supplied URLs, no fabricated latest dates.
from urllib.parse import urlparse

CELLAR_SPARQL = "https://publications.europa.eu/webapi/rdf/sparql"
CELLAR_CELEX_BASE = "https://publications.europa.eu/resource/celex/"
CELLAR_TIMEOUT = (10, 45)
MAX_SPARQL_BYTES = 2_000_000
MAX_VERSIONS = 100


def _base_record(regulation):
    law = LAW_REGISTRY.get(regulation, {})
    return {
        "regulation": regulation, "basic_celex": law.get("celex"),
        "snapshot_date": law.get("snapshot_date"),
        "snapshot_file": law.get("snapshot_file"),
        "official_basic_url": law.get("official_basic_url"),
        "official_metadata_endpoint": CELLAR_SPARQL,
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "status": "unverified", "source": "EU Publications Office / Cellar SPARQL",
        "latest_date_candidate": None, "latest_celex_candidate": None,
        "observed_consolidations": [], "date_comparison": "not_verified",
        "article_level_changes_verified": False,
        "application_date_verified": False,
        "current_legal_status_verified": False,
        "complete_consolidation_list_verified": False,
    }


def _version_celex_pattern(basic_celex):
    # Sector 3 basic act -> sector 0 consolidated versions.
    return re.compile(r"^0" + re.escape(basic_celex[1:]) + r"-(\d{8})$")


def _sparql_versions_query(celex, mode="relation"):
    assert re.fullmatch(r"3\d{4}R\d{4}", celex), "Only allowlisted regulation CELEX IDs"
    prefix = "0" + celex[1:] + "-"
    header = "PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>\n"
    if mode == "relation":
        # Both documented relation predicates; incoming edges point at base work.
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?original cdm:resource_legal_id_celex "{celex}" .
  {{ ?consolidated cdm:act_consolidated_based_on_resource_legal ?original . }}
  UNION
  {{ ?consolidated cdm:act_consolidated_consolidates_resource_legal ?original . }}
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(STRSTARTS(STR(?consolidated_celex), "{prefix}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    if mode == "celex_index":
        # Validated against live Cellar: the plain SPARQL CONTAINS filter
        # returned 19 CELEX records (including 02019R1793-20260630).
        # The old Virtuoso bif:contains clause silently returned zero rows.
        # Search broadly, then use _parse_versions() to accept ONLY the
        # correct sector-0 consolidated CELEX IDs with valid dates.
        search_fragment = celex[1:]  # e.g. 2019R1793
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(CONTAINS(STR(?consolidated_celex), "{search_fragment}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    raise ValueError("Unknown SPARQL query mode")


def _run_cellar_sparql(query, session=None):
    client = session if session is not None else requests.Session()
    response = client.get(
        CELLAR_SPARQL, params={"query": query},
        headers={"Accept": "application/sparql-results+json",
                 "User-Agent": "ScientificAIBuilder-Research/0.2"},
        timeout=CELLAR_TIMEOUT, allow_redirects=True,
    )
    resolved = getattr(response, "url", CELLAR_SPARQL)
    parsed = urlparse(resolved)
    if parsed.scheme != "https" or parsed.hostname != "publications.europa.eu":
        raise ValueError("untrusted_redirect")
    if response.status_code != 200:
        raise ValueError(f"http_{response.status_code}")
    raw = response.content
    if not raw or len(raw) > MAX_SPARQL_BYTES:
        raise ValueError("empty_or_oversize_response")
    # Virtuoso can return HTTP 200 with plain-text query error.
    content_type = response.headers.get("Content-Type", "").lower()
    if "json" not in content_type:
        raise ValueError("non_json_or_sparql_error")
    data = response.json()
    if not isinstance(data.get("results", {}).get("bindings"), list):
        raise ValueError("invalid_sparql_result_structure")
    return data["results"]["bindings"]


def _parse_versions(bindings, basic_celex, as_of=None):
    as_of = as_of or datetime.now(timezone.utc).date()
    pattern = _version_celex_pattern(basic_celex)
    found = {}
    for row in bindings:
        value = row.get("consolidated_celex", {}).get("value", "")
        match = pattern.fullmatch(value)
        if not match:
            continue
        try:
            dt = datetime.strptime(match.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        # Future-dated consolidations are recorded separately, not current.
        if dt > as_of:
            continue
        found[value] = {
            "celex": value,
            "date": dt.isoformat(),
            "official_celex_url": CELLAR_CELEX_BASE + value,
        }
    return sorted(found.values(), key=lambda v: (v["date"],v["celex"]), reverse=True)


def retrieve_cellar_current_law(regulation, session=None, as_of=None):
    """Official metadata check only; NEVER certify legal force or amended clauses.

    Relational lookup + independently scoped CELEX-index lookup.
    Missing results or failed confirmation are UNVERIFIED, not 'no update'.
    """
    rec = _base_record(regulation)
    if regulation not in LAW_REGISTRY:
        rec["error"] = "not_allowlisted"
        return rec
    basic = LAW_REGISTRY[regulation]["celex"]
    as_of = as_of or datetime.now(timezone.utc).date()
    if isinstance(as_of, str):
        as_of = date.fromisoformat(as_of)
    rec["as_of_date"] = as_of.isoformat()

    try:
        rel = _run_cellar_sparql(_sparql_versions_query(basic,"relation"),session)
        rec["relation_row_count"] = len(rel)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        rel = []
        rec["relation_error"] = f"{type(exc).__name__}: {exc}"

    # Important: index query is independent, not assumed equivalent to links.
    try:
        idx = _run_cellar_sparql(_sparql_versions_query(basic,"celex_index"),session)
        rec["celex_index_row_count"] = len(idx)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        idx = []
        rec["celex_index_error"] = f"{type(exc).__name__}: {exc}"

    observed = _parse_versions(rel+idx,basic,as_of=as_of)
    rec["observed_consolidations"] = observed
    rec["lookup_methods"] = ["relation", "celex_index"]
    if not observed:
        rec["error"] = "no_verified_consolidation_rows"
        return rec

    candidate = observed[0]
    rec["latest_date_candidate"] = candidate["date"]
    rec["latest_celex_candidate"] = candidate["celex"]
    rec["latest_candidate_official_url"] = candidate["official_celex_url"]

    # If either query failed, retrieval may be incomplete.
    if "relation_error" in rec or "celex_index_error" in rec:
        rec["status"] = "partial_metadata"
        rec["date_comparison"] = "latest_observed_only_incomplete_verification"
        return rec
    if len(rel) >= MAX_VERSIONS or len(idx) >= MAX_VERSIONS:
        rec["status"] = "partial_metadata"
        rec["error"] = "result_limit_reached"
        return rec

    rec["status"] = "metadata_found"
    rec["complete_consolidation_list_verified"] = False  # Graph freshness/coverage not provable here.
    snapshot = rec["snapshot_date"]
    if snapshot is None:
        rec["date_comparison"] = "original_snapshot_without_consolidation_date"
    elif candidate["date"] > snapshot:
        rec["date_comparison"] = "newer_consolidation_observed"
    elif candidate["date"] == snapshot:
        rec["date_comparison"] = "matches_latest_observed_consolidation"
    else:
        rec["date_comparison"] = "latest_observed_older_than_snapshot_review"
    return rec

# Alias preserves existing tool wiring below.
retrieve_eurlex_basic_page = retrieve_cellar_current_law

## 5. Offline Cellar parser tests
No internet, no OpenAI API. Confirm metadata parsing / conservative `unverified` behavior.

In [8]:
# Offline tests: these DO NOT establish real version metadata.
class _FakeResponse:
    def __init__(self, rows, status=200, url=CELLAR_SPARQL, bad_content_type=False):
        self.status_code = status
        self.url = url
        self.headers = {"Content-Type": "text/html" if bad_content_type else "application/sparql-results+json"}
        self._payload = {"head":{"vars":["consolidated_celex"]},"results":{"bindings":rows}}
        self.content = json.dumps(self._payload).encode("utf-8")
    def json(self): return self._payload

class _FakeSession:
    def __init__(self, responses):
        self.responses = list(responses)
    def get(self,*args,**kwargs):
        return self.responses.pop(0)

def _binding(celex):
    return {"consolidated_celex":{"type":"literal","value":celex}}

valid = _FakeSession([
    _FakeResponse([_binding("02019R1793-20260401")]),
    _FakeResponse([_binding("02019R1793-20260630"),_binding("02019R1793-20260401")]),
])
r = retrieve_cellar_current_law("2019/1793",session=valid,as_of="2026-10-19")
assert r["status"] == "metadata_found" and r["latest_date_candidate"] == "2026-06-30"
assert r["date_comparison"] == "matches_latest_observed_consolidation"
assert r["current_legal_status_verified"] is False

empty = _FakeSession([_FakeResponse([]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=empty,as_of="2026-10-19")
assert r["status"] == "unverified" and r["latest_date_candidate"] is None

blocked = _FakeSession([_FakeResponse([],status=202),_FakeResponse([],status=202)])
r = retrieve_cellar_current_law("2019/1793",session=blocked,as_of="2026-10-19")
assert r["status"] == "unverified"

wrong_law = _FakeSession([_FakeResponse([_binding("02023R0915-20260819")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=wrong_law,as_of="2026-10-19")
assert r["status"] == "unverified"

future = _FakeSession([_FakeResponse([_binding("02019R1793-20270101")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=future,as_of="2026-10-19")
assert r["status"] == "unverified"

assert retrieve_cellar_current_law("2099/9999")["error"] == "not_allowlisted"
print("PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior")

# Regression: use the successful Cellar SPARQL predicate, not Virtuoso full-text bif:contains.
index_query = _sparql_versions_query("32019R1793", "celex_index")
assert 'FILTER(CONTAINS(STR(?consolidated_celex), "2019R1793"))' in index_query
assert "bif:contains" not in index_query
print("PASS: CELEX index query matches successful live diagnostic")

PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior
PASS: CELEX index query matches successful live diagnostic


## 6. Optional live probe — before paid runs
The Cellar probe costs **no OpenAI tokens**, but uses the internet. A valid probe for 2019/1793 checks connectivity, not coverage for every regulation. Failed probe should block paid runs if you are evaluating live verification.

In [10]:
RUN_LIVE_PROBE =True  # enable once tomorrow; requires internet
NB08_PROBE_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb08_cellar_probe.json")
if RUN_LIVE_PROBE:
    live_probe = retrieve_cellar_current_law("2019/1793")
    NB08_PROBE_PATH.write_text(json.dumps(live_probe, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Probe:", live_probe["status"], live_probe["date_comparison"], "Saved:", NB08_PROBE_PATH)
else:
    print("Live probe SKIPPED (no network request)")

Probe: metadata_found matches_latest_observed_consolidation Saved: data\build03_nb08_cellar_probe.json


## 7. Build03-B Function Tools and full evidence logs
Every legal search logs query and full retrieved chunks. Each current-law call logs the complete returned metadata. The agent must decide whether to invoke current-law verification: **do not force it for every Q**. Tool registration alone does not call any API.

In [11]:
TOOL_LOGS = {"search": [], "current_law": []}

@function_tool
def search_regulation(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law library for relevant provisions.

    Args:
        query: Focused legal query.
        top_k: Number of result chunks, 1–10.
    """
    results = hybrid_search(query, top_k=max(1, min(int(top_k), 10)))
    TOOL_LOGS["search"].append({"query": query, "top_k": top_k, "results": results})
    return json.dumps([{k: r.get(k) for k in ["rank", "chunk_id", "file", "article", "location", "heading", "text"]} for r in results], ensure_ascii=False)

@function_tool
def check_current_law(regulation: str) -> str:
    """Query official EU Cellar metadata for the latest observed consolidation of an allowlisted regulation.

    Args:
        regulation: EU regulation, e.g. 2019/1793 or 2023/915.
    """
    rec = retrieve_cellar_current_law(regulation.strip())
    TOOL_LOGS["current_law"].append(rec)
    return json.dumps(rec, ensure_ascii=False)

SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]
MATCHED_RULES = """Use available tools to ground legal claims in the frozen 5-law corpus.
When available, official Cellar consolidation metadata can supplement local evidence; when unavailable,
state that live currentness was not verified, rather than implying that dates are current.
Identify provisions requiring checks for amendments, entry into force, application date,
transitional rules, annex changes and product-specific classification.
Never infer effective article-level law solely from a matching consolidation date.
Distinguish verified observations from legal conclusions. No invented limits, rates or requirements.
Use 3 headings: 1. Short answer; 2. Legal basis and relevant conditions;
3. Uncertainties / verification needed."""
INSTRUCTIONS = "\n\n".join([SHARED_CONTEXT, SHARED_SYSTEM_PROMPT, MATCHED_RULES])
agent_B = Agent(
    name="Build03-B Full Benchmark",
    model=MODEL_NAME,
    model_settings=ModelSettings(reasoning={"effort": REASONING_EFFORT}, max_tokens=int(nb04b_config["max_output_tokens"])),
    instructions=INSTRUCTIONS,
    tools=[search_regulation, check_current_law],
)
print("Tools ready:", "search_regulation", "check_current_law")
print("Agent ready:", MODEL_NAME, REASONING_EFFORT, "| benchmark questions:", len(questions))

Tools ready: search_regulation check_current_law
Agent ready: gpt-6-luna medium | benchmark questions: 10


## 8. Run plan: safe, resumable, paid calls OFF by default

**Tomorrow:** first run setup/parity, then set `RUN_LIVE_PROBE=True` and verify `metadata_found`. Next set `RUN_OFFICIAL=True`, initially `SELECT_QIDS=["Q1"]` for a smoke test. Once it succeeds, change to `SELECT_QIDS=list(questions)` and rerun: completed output files are skipped. Each QID is saved immediately via exclusive file creation. Errors are saved separately so a failed Q does not erase other runs.

**Warning:** Resuming an existing run skips output files; do not mix different model settings or instructions in one experiment. A manifest fingerprint protects against accidental incompatible runs. **A rerun with different settings needs a new output directory/experiment ID.**

In [15]:
RUN_OFFICIAL = True  # <-- change tomorrow ONLY after verifying setup and probe
SELECT_QIDS =  list(questions)  # first smoke test; then list(questions) for Q1..Q10
REQUIRE_LIVE_PROBE = True
MAX_TURNS = 18
EXPERIMENT_ID = "NB08_build03B_full_v1"
RUN_DIR = __import__("pathlib").Path("results/build03")
assert all(q in questions for q in SELECT_QIDS)
assert len(set(SELECT_QIDS)) == len(SELECT_QIDS)

# Fixed experimental settings fingerprint: do not change mid-batch.
settings = {
    "experiment_id": EXPERIMENT_ID,
    "benchmark_sha256": benchmark_hash,
    "model": MODEL_NAME,
    "reasoning_effort": REASONING_EFFORT,
    "max_output_tokens": int(nb04b_config["max_output_tokens"]),
    "instructions_sha256": hashlib.sha256(INSTRUCTIONS.encode()).hexdigest(),
    "retrieval_config_sha256": hashlib.sha256(FILES["retrieval_config"].read_bytes()).hexdigest(),
    "chunks_sha256": hashlib.sha256(FILES["chunks"].read_bytes()).hexdigest(),
    "max_turns": MAX_TURNS,
    "tools": ["search_regulation", "check_current_law"],
}
RUN_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RUN_DIR / "manifest.json"
if MANIFEST_PATH.exists():
    previous = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    assert previous == settings, "STOP: existing run directory has different settings; do NOT mix experiments"
else:
    with MANIFEST_PATH.open("x", encoding="utf-8") as f:
        json.dump(settings, f, indent=2, ensure_ascii=False)
print("Plan:", SELECT_QIDS, "| paid run:", RUN_OFFICIAL)
print("Output directory:", RUN_DIR)
print("Existing completed:", sorted(p.stem for p in RUN_DIR.glob("Q*.json") if not p.stem.endswith("_error")))

Plan: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10'] | paid run: True
Output directory: data\build03_nb08_B_official_runs
Existing completed: ['Q1']


## 9. Execute per-QID (requires OpenAI API; can also call live Cellar)
Jupyter supports top-level `await Runner.run(...)`. Keep `RUN_OFFICIAL=False` while reading. A completed QID is never overwritten. Tool-call data is cleared between questions; save failures to a separate `*_error.json` diagnostic.

In [16]:
async def execute_one_qid(qid):
    assert PARITY_OK and matched == 50
    assert qid in questions
    output_file = RUN_DIR / f"{qid}.json"
    if output_file.exists():
        print("SKIP already saved:", qid)
        return None
    TOOL_LOGS["search"].clear()
    TOOL_LOGS["current_law"].clear()
    started = datetime.now(timezone.utc).isoformat()
    try:
        result = await Runner.run(agent_B, questions[qid], max_turns=MAX_TURNS)
        record = {
            "experiment_id": EXPERIMENT_ID, "arm": "Build03-B", "qid": qid,
            "question": questions[qid], "benchmark_id": benchmark["benchmark_id"],
            "benchmark_sha256": benchmark_hash, "settings_sha256": hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(),
            "model": MODEL_NAME, "reasoning_effort": REASONING_EFFORT,
            "tools_available": ["search_regulation", "check_current_law"],
            "retrieval_parity": {"matched": matched, "total": 50},
            "search_calls": list(TOOL_LOGS["search"]),
            "current_law_calls": list(TOOL_LOGS["current_law"]),
            "final_output": str(result.final_output),
            "started_utc": started, "completed_utc": datetime.now(timezone.utc).isoformat(),
            "status": "evaluation_pending",
        }
        with output_file.open("x", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f"SAVED {qid}: search={len(record['search_calls'])}, current_law={len(record['current_law_calls'])}")
        return record
    except Exception as exc:
        error_path = RUN_DIR / f"{qid}_error.json"
        details = {
            "qid": qid, "question": questions[qid], "started_utc": started,
            "failed_utc": datetime.now(timezone.utc).isoformat(),
            "error_type": type(exc).__name__, "error_message": str(exc),
            "partial_search_calls": list(TOOL_LOGS["search"]),
            "partial_current_law_calls": list(TOOL_LOGS["current_law"]),
        }
        # Diagnostics only; retain the latest failure without touching successful answers.
        error_path.write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")
        print("FAILED", qid, type(exc).__name__, str(exc)[:350], "| diagnostics:", error_path)
        return None

if RUN_OFFICIAL:
    assert PARITY_OK, "STOP: parity failed"
    if REQUIRE_LIVE_PROBE:
        assert NB08_PROBE_PATH.exists(), "STOP: run the live Cellar probe before paid evaluation"
        saved_probe = json.loads(NB08_PROBE_PATH.read_text(encoding="utf-8"))
        assert saved_probe.get("status") == "metadata_found", "STOP: live Cellar probe is unverified"
        assert saved_probe.get("date_comparison") == "matches_latest_observed_consolidation", "STOP: review metadata mismatch"
        # Probe is a connectivity check; verify it again on the day of a resumed run.
        live_probe = retrieve_cellar_current_law("2019/1793")
        assert live_probe["status"] == "metadata_found", "STOP: current Cellar connectivity check failed"
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (hidden): ").strip()
    assert os.getenv("OPENAI_API_KEY"), "Missing API key"
    for qid in SELECT_QIDS:
        await execute_one_qid(qid)
else:
    print("OFFICIAL RUN DISABLED. No paid OpenAI API requests.")

SKIP already saved: Q1
SAVED Q2: search=1, current_law=1
SAVED Q3: search=2, current_law=1
SAVED Q4: search=4, current_law=1
SAVED Q5: search=8, current_law=2
SAVED Q6: search=6, current_law=1
SAVED Q7: search=5, current_law=2
SAVED Q8: search=5, current_law=5
SAVED Q9: search=9, current_law=4
SAVED Q10: search=9, current_law=5


## 10. Result viewer / summary CSV
Review Build02 frozen answer and Build03-B output side-by-side. This section is **read-only** for Build02 and does not judge correctness automatically. Counts come from saved tool logs, not inferred from answer text. `current_law` counts include metadata failures too.

In [17]:
rows = []
for qid in questions:
    baseline = comparison_df.loc[comparison_df.qid == qid, "rag_answer"].iloc[0]
    path = RUN_DIR / f"{qid}.json"
    obj = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    law_calls = obj.get("current_law_calls", []) if obj else []
    rows.append({
        "qid": qid, "question": questions[qid],
        "baseline_available": True,
        "build03_status": "saved" if obj else "not_run",
        "search_calls": len(obj.get("search_calls", [])) if obj else None,
        "current_law_calls": len(law_calls) if obj else None,
        "metadata_found_calls": sum(x.get("status") == "metadata_found" for x in law_calls) if obj else None,
        "unverified_or_partial_calls": sum(x.get("status") != "metadata_found" for x in law_calls) if obj else None,
        "baseline_chars": len(str(baseline)),
        "build03_chars": len(obj["final_output"]) if obj else None,
    })
summary_df = pd.DataFrame(rows)
SUMMARY_PATH = RUN_DIR / "q1_q10_run_summary.csv"
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
display(summary_df[["qid", "build03_status", "search_calls", "current_law_calls", "metadata_found_calls", "unverified_or_partial_calls"]])
print("Run summary saved:", SUMMARY_PATH)
print("Completed QIDs:", int(summary_df.build03_status.eq("saved").sum()), "/ 10")

def show_comparison(qid="Q1"):
    assert qid in questions
    print("="*85, "\n", qid, questions[qid])
    print("\n--- FROZEN BUILD02 RAG ---\n")
    print(comparison_df.loc[comparison_df.qid == qid, "rag_answer"].iloc[0])
    path = RUN_DIR / f"{qid}.json"
    print("\n--- BUILD03-B ---\n")
    if not path.exists():
        print("NOT RUN")
        return
    obj = json.loads(path.read_text(encoding="utf-8"))
    print(obj["final_output"])
    print("\n--- TOOL TRACE SUMMARY ---")
    for c in obj.get("search_calls", []):
        print("search:", c["query"], "| chunk IDs:", [r["chunk_id"] for r in c["results"]])
    for c in obj.get("current_law_calls", []):
        print("current:", c.get("regulation"), c.get("status"), c.get("date_comparison"))

show_comparison("Q1")

,qid,build03_status,search_calls,current_law_calls,metadata_found_calls,unverified_or_partial_calls
0,Q1,saved,4,2,2,0
1,Q2,saved,1,1,1,0
2,Q3,saved,2,1,1,0
3,Q4,saved,4,1,1,0
4,Q5,saved,8,2,2,0
5,Q6,saved,6,1,1,0
6,Q7,saved,5,2,2,0
7,Q8,saved,5,5,4,1
8,Q9,saved,9,4,4,0
9,Q10,saved,9,5,4,1


Run summary saved: data\build03_nb08_B_official_runs\q1_q10_run_summary.csv
Completed QIDs: 10 / 10
 Q1 A shipment of dried figs originating from Türkiye is found to contain aflatoxin B1. Which EU legislation is relevant to this case?

--- FROZEN BUILD02 RAG ---

## 1. Short answer
The supplied evidence identifies two relevant EU acts:

- **Commission Regulation (EU) 2023/915**, which sets maximum levels for aflatoxins in dried figs. [E2]
- **Commission Implementing Regulation (EU) 2019/1793**, which lists dried figs originating in Türkiye for increased official controls concerning aflatoxins and ochratoxin A. [E1]

## 2. Legal basis and relevant conditions
Under the cited entry in Regulation (EU) 2023/915, the maximum level for aflatoxin B1 in dried figs is **6.0 μg/kg**; the maximum for the sum of aflatoxins B1, B2, G1 and G2 is **10.0 μg/kg**. The entry also addresses products consisting only of dried figs and processed products containing at least 80% dried figs. [E2]

The cited en

## 11. Manual reference-based evaluation worksheet
**Do not auto-grade.** Use the pre-existing frozen Notion Verified Reference **Q1–Q10** and trace each legal claim back to retrieved chunks. The structured worksheet is a human review template. Q7: border Art 66 + related actions and post-market Art 19; Q8: version-sensitive scope/annex/entry/transition provisions. No rubrics or must-have answer hints are passed into the Agent.

Keep `Correct`, `Grounded`, `Complete` independent from `CurrentLawTool` results. Do not mark an answer Yes solely because it called both tools.

In [18]:
EVAL_PATH = RUN_DIR / "q1_q10_human_evaluation.csv"
EVAL_COLS = ["qid", "question", "build", "correct", "grounded", "complete",
             "critical_legal_error", "reference_must_have_covered", "reference_must_have_missing",
             "evidence_chunk_ids_checked", "unsupported_claims", "citation_traceability",
             "current_law_use_appropriate", "metadata_vs_applicability_disclaimer",
             "uncertainty_handling", "failure_types", "notes", "reviewed_utc"]
if EVAL_PATH.exists():
    print("PRESERVED existing evaluation:", EVAL_PATH)
else:
    eval_rows = []
    for qid, question in questions.items():
        for build in ["Build02 frozen", "Build03-B"]:
            eval_rows.append({**{k: "" for k in EVAL_COLS},
                              "qid": qid, "question": question, "build": build,
                              "correct": "REVIEW", "grounded": "REVIEW", "complete": "REVIEW"})
    pd.DataFrame(eval_rows, columns=EVAL_COLS).to_csv(EVAL_PATH, index=False, encoding="utf-8-sig")
    print("Created evaluation worksheet:", EVAL_PATH)
print("Valid grade values: Yes / Partial / No; REVIEW means not evaluated")

Created evaluation worksheet: data\build03_nb08_B_official_runs\q1_q10_human_evaluation.csv
Valid grade values: Yes / Partial / No; REVIEW means not evaluated


## 12. Build04 Gap Analysis — map observed failures to proposed capabilities
Populate only **after human evidence review**. A candidate Tool is *not* justified by a single vague answer; log relevant QID, retrieved evidence and exact failure. This sheet is separate from the frozen benchmark and does **not** feed back into these Agent runs.

In [ ]:
GAP_PATH = RUN_DIR / "build04_gap_analysis.csv"
GAP_FIELDS = ["qid", "observed_failure", "failure_category", "supporting_chunk_or_call_ids",
              "impact", "proposed_build04_change", "is_new_tool_needed", "priority", "notes"]
CATEGORIES = ["Retrieval", "Current Law", "Reasoning", "Missing Facts", "Tool Routing", "Citation/Evidence", "Other"]
PROPOSED_CAPABILITIES = {
    "Retrieval": "Evaluate exact provision lookup (Article/Annex) if exact text was missed",
    "Current Law": "Enhance amendment/application-date and original-act verification only where needed",
    "Reasoning": "Investigate staged evidence-validation workflow",
    "Missing Facts": "Structured product/shipment fact collection before deciding",
    "Tool Routing": "Tool selection rules, stop conditions and budget",
    "Citation/Evidence": "Provision-level evidence links and traceable claim validation",
}
if GAP_PATH.exists():
    print("PRESERVED existing gap analysis:", GAP_PATH)
else:
    pd.DataFrame(columns=GAP_FIELDS).to_csv(GAP_PATH, index=False, encoding="utf-8-sig")
    print("Created blank Build04 gap analysis:", GAP_PATH)
print("Failure categories:", ", ".join(CATEGORIES))
print("Candidate changes are hypotheses, not predefined model upgrades.")

## 13. Completion gate and interpretation
1. Confirm 50/50 ordered parity and same frozen corpus / benchmark fingerprints.
2. Complete **ten** Build03-B QID files, or explicitly document omissions and errors.
3. Review Q1–Q10 against the frozen Verified Reference; mark *Correct / Grounded / Complete* per arm; link actual evidence chunks.
4. Inspect Tool logs: call count, redundant queries, correct invocation of Cellar, all `unverified` outcomes.
5. Populate Build04 gap sheet with **observed** evidence, distinguish required feature from optional enhancement.
6. Freeze NB08 results before changing prompts or adding Build04 tools; then design Build04 workflow and re-run the same benchmark (plus new practical cases if needed).

**Interpretation guardrail:** Q8 capability demonstration ≠ proven global accuracy increase. Currentness metadata ≠ article-level legal force. More Tool calls ≠ better answer. Build02 vs B is a stage comparison, not a controlled isolation of agent architecture (NB07 is the narrower ablation).